# Microsoft Fabric Runtime migration: 1.2 to 1.3 or 2.0

This notebook inventories Fabric workspace defaults and Environment items that still use Runtime 1.2, then prepares or executes their migration through the public Fabric REST APIs.

> **Safety first:** migration runs in dry-run mode by default. Write operations require an explicit workspace allowlist, `EXECUTE_MIGRATION = True`, and a matching confirmation phrase.

## What the notebook changes

- The default Spark runtime in each selected workspace.
- The runtime of each selected Environment item, followed by an Environment publish.
- Nothing outside the explicit workspace allowlist when execution is enabled.

The notebook does **not** prove that notebooks, Spark Job Definitions, custom wheels, JARs, or Delta features are compatible with the target runtime. Validate those dependencies in a development workspace before production rollout.

## Prerequisites

- Run from a Microsoft Fabric notebook with `sempy` available.
- Use an identity that can read the target workspaces and has Admin/write permissions for changes.
- Review Environment library compatibility before migration.
- Keep `EXECUTE_MIGRATION = False` for the first run.

## Recommended rollout

1. Inventory every accessible workspace and Environment.
2. Choose Runtime 1.3 for the lower-change transition path, or Runtime 2.0 after Spark 4 / Python 3.13 / Java 21 / Scala 2.13 compatibility tests.
3. Test one development workspace and its attached workloads.
4. Expand the allowlist by environment: DEV, then UAT, then PROD.
5. Validate representative notebooks and Spark Job Definitions after each wave.

---

# 1. Inventory

## 1.a. Workspace runtime inventory

Lists every accessible workspace whose default Spark runtime can be read, then isolates Runtime 1.2 workspaces. Read failures stay visible in the result instead of being silently ignored.

In [ ]:
import pandas as pd
import sempy.fabric as fabric

client = fabric.FabricRestClient()
workspace_results = []
workspaces = fabric.list_workspaces()

for _, workspace in workspaces.iterrows():
    workspace_id = workspace["Id"]
    workspace_name = workspace["Name"]

    try:
        response = client.get(
            f"/v1/workspaces/{workspace_id}/spark/settings"
        )

        if response.status_code == 200:
            runtime = (
                response.json()
                .get("environment", {})
                .get("runtimeVersion")
            )
            status = "Read"
            detail = ""
        else:
            runtime = None
            status = f"HTTP {response.status_code}"
            detail = response.text[:500]
    except Exception as exception:
        runtime = None
        status = "Failed"
        detail = str(exception)

    workspace_results.append(
        {
            "WorkspaceName": workspace_name,
            "WorkspaceId": workspace_id,
            "RuntimeVersion": runtime,
            "Status": status,
            "Detail": detail,
        }
    )

workspace_df = pd.DataFrame(
    workspace_results,
    columns=[
        "WorkspaceName",
        "WorkspaceId",
        "RuntimeVersion",
        "Status",
        "Detail",
    ],
)

runtime12_workspaces = workspace_df[
    workspace_df["RuntimeVersion"] == "1.2"
]

print(f"Accessible workspaces: {len(workspace_df)}")
print(f"Workspaces on Runtime 1.2: {len(runtime12_workspaces)}")
display(workspace_df)

## 1.b. Environment runtime inventory

Lists Environment items and their effective runtime. The migration cell later uses the stable `beta=false` contracts and refuses to publish an Environment that already has unpublished compute changes.

In [ ]:
def list_workspace_environments(workspace_id):
    environments = []
    next_path = f"/v1/workspaces/{workspace_id}/environments"

    while next_path:
        response = client.get(next_path)
        if response.status_code != 200:
            raise RuntimeError(
                f"HTTP {response.status_code}: {response.text[:500]}"
            )

        payload = response.json()
        environments.extend(payload.get("value", []))
        next_path = payload.get("continuationUri")

    return environments


environment_results = []

for _, workspace in workspaces.iterrows():
    workspace_id = workspace["Id"]
    workspace_name = workspace["Name"]

    try:
        environments = list_workspace_environments(workspace_id)
    except Exception as exception:
        environment_results.append(
            {
                "WorkspaceName": workspace_name,
                "WorkspaceId": workspace_id,
                "EnvironmentName": None,
                "EnvironmentId": None,
                "RuntimeVersion": None,
                "PublishState": None,
                "Status": "Failed",
                "Detail": str(exception),
            }
        )
        continue

    for environment in environments:
        environment_id = environment["id"]
        environment_name = environment["displayName"]
        publish_state = (
            environment.get("properties", {})
            .get("publishDetails", {})
            .get("state")
        )

        try:
            response = client.get(
                f"/v1/workspaces/{workspace_id}/environments/"
                f"{environment_id}/sparkcompute?beta=false"
            )

            if response.status_code == 200:
                runtime = response.json().get("runtimeVersion")
                status = "Read"
                detail = ""
            else:
                runtime = None
                status = f"HTTP {response.status_code}"
                detail = response.text[:500]
        except Exception as exception:
            runtime = None
            status = "Failed"
            detail = str(exception)

        environment_results.append(
            {
                "WorkspaceName": workspace_name,
                "WorkspaceId": workspace_id,
                "EnvironmentName": environment_name,
                "EnvironmentId": environment_id,
                "RuntimeVersion": runtime,
                "PublishState": publish_state,
                "Status": status,
                "Detail": detail,
            }
        )

environment_df = pd.DataFrame(
    environment_results,
    columns=[
        "WorkspaceName",
        "WorkspaceId",
        "EnvironmentName",
        "EnvironmentId",
        "RuntimeVersion",
        "PublishState",
        "Status",
        "Detail",
    ],
)

runtime12_environments = environment_df[
    environment_df["RuntimeVersion"] == "1.2"
]

print(f"Environment items: {len(environment_df)}")
print(f"Environments on Runtime 1.2: {len(runtime12_environments)}")
display(environment_df)

# 2. Migration plan and execution

The next cell is autonomous and starts in **dry-run mode**.

1. Set `TARGET_RUNTIME` to `"1.3"` or `"2.0"`.
2. Keep `TARGET_WORKSPACES` empty to inventory all accessible workspaces in dry run.
3. Review rows marked `Planned`, `Skipped`, or `Failed`.
4. For execution, populate `TARGET_WORKSPACES`, set `EXECUTE_MIGRATION = True`, and set `CONFIRMATION` to the exact phrase printed by the configuration (`UPGRADE TO 1.3` or `UPGRADE TO 2.0`).
5. Re-run the cell and validate representative workloads after each workspace wave.

Environment publishes are asynchronous. The notebook waits for a terminal publish state and then reads the effective runtime again before reporting `Verified`.

In [ ]:
import time
from copy import deepcopy

import pandas as pd
import sempy.fabric as fabric

SOURCE_RUNTIME = "1.2"
TARGET_RUNTIME = "2.0"  # Use "1.3" for the lower-risk transition path.
TARGET_WORKSPACES = [
    # "Fabric-DEV",
    # "Fabric-UAT",
    # "Fabric-PROD",
]
EXECUTE_MIGRATION = False
CONFIRMATION = ""

PUBLISH_TIMEOUT_SECONDS = 1800
PUBLISH_POLL_SECONDS = 15
MAX_RATE_LIMIT_RETRIES = 5
SUPPORTED_TARGET_RUNTIMES = {"1.3", "2.0"}

if TARGET_RUNTIME not in SUPPORTED_TARGET_RUNTIMES:
    raise ValueError(
        f"TARGET_RUNTIME must be one of {sorted(SUPPORTED_TARGET_RUNTIMES)}"
    )

if EXECUTE_MIGRATION and not TARGET_WORKSPACES:
    raise ValueError(
        "Set TARGET_WORKSPACES explicitly before enabling EXECUTE_MIGRATION."
    )

expected_confirmation = f"UPGRADE TO {TARGET_RUNTIME}"
if EXECUTE_MIGRATION and CONFIRMATION != expected_confirmation:
    raise ValueError(
        f'Set CONFIRMATION to "{expected_confirmation}" before execution.'
    )

client = fabric.FabricRestClient()
migration_results = []


def call_api(method, path, **kwargs):
    for attempt in range(MAX_RATE_LIMIT_RETRIES + 1):
        response = getattr(client, method)(path, **kwargs)
        if response.status_code != 429:
            return response

        if attempt == MAX_RATE_LIMIT_RETRIES:
            return response

        retry_after = int(response.headers.get("Retry-After", "5"))
        time.sleep(min(retry_after, 120))


def response_error(response):
    try:
        payload = response.json()
        return payload.get("message") or payload.get("errorCode") or str(payload)
    except Exception:
        return response.text


def get_json(path):
    response = call_api("get", path)
    if response.status_code != 200:
        raise RuntimeError(
            f"GET {path} failed ({response.status_code}): "
            f"{response_error(response)}"
        )
    return response.json()


def get_paginated_values(path):
    values = []
    next_path = path

    while next_path:
        payload = get_json(next_path)
        values.extend(payload.get("value", []))
        next_path = payload.get("continuationUri")

    return values


def select_workspaces():
    available = fabric.list_workspaces()

    if not TARGET_WORKSPACES:
        return available

    selected = available[available["Name"].isin(TARGET_WORKSPACES)].copy()
    missing = sorted(set(TARGET_WORKSPACES) - set(selected["Name"]))

    if missing:
        raise ValueError(
            "These TARGET_WORKSPACES are not accessible or do not exist: "
            + ", ".join(missing)
        )

    return selected


def get_workspace_runtime(workspace_id):
    payload = get_json(f"/v1/workspaces/{workspace_id}/spark/settings")
    return payload.get("environment", {}).get("runtimeVersion")


def list_environments(workspace_id):
    return get_paginated_values(
        f"/v1/workspaces/{workspace_id}/environments"
    )


def get_environment_compute(workspace_id, environment_id, staging=False):
    state = "/staging" if staging else ""
    return get_json(
        f"/v1/workspaces/{workspace_id}/environments/"
        f"{environment_id}{state}/sparkcompute?beta=false"
    )


def get_environment_metadata(workspace_id, environment_id):
    return get_json(
        f"/v1/workspaces/{workspace_id}/environments/{environment_id}"
    )


def comparable_compute(compute):
    fields = (
        "instancePool",
        "driverCores",
        "driverMemory",
        "executorCores",
        "executorMemory",
        "dynamicExecutorAllocation",
        "sparkProperties",
        "runtimeVersion",
    )
    return {field: deepcopy(compute.get(field)) for field in fields}


def publish_state(metadata):
    return (
        metadata.get("properties", {})
        .get("publishDetails", {})
        .get("state")
    )


def wait_for_environment_publish(workspace_id, environment_id):
    deadline = time.monotonic() + PUBLISH_TIMEOUT_SECONDS
    terminal_states = {"Success", "Failed", "Cancelled"}

    while time.monotonic() < deadline:
        metadata = get_environment_metadata(workspace_id, environment_id)
        state = publish_state(metadata)

        if state in terminal_states:
            return state

        time.sleep(PUBLISH_POLL_SECONDS)

    return "TimedOut"


def add_result(
    workspace_name,
    scope,
    item_name,
    item_id,
    before,
    after,
    status,
    detail="",
):
    migration_results.append(
        {
            "WorkspaceName": workspace_name,
            "Scope": scope,
            "ItemName": item_name,
            "ItemId": item_id,
            "RuntimeBefore": before,
            "RuntimeAfter": after,
            "Status": status,
            "Detail": detail,
        }
    )


def migrate_workspace(workspace_name, workspace_id):
    try:
        current_runtime = get_workspace_runtime(workspace_id)

        if current_runtime != SOURCE_RUNTIME:
            add_result(
                workspace_name,
                "Workspace",
                workspace_name,
                workspace_id,
                current_runtime,
                current_runtime,
                "Skipped",
                f"Only Runtime {SOURCE_RUNTIME} is migrated.",
            )
            return

        if not EXECUTE_MIGRATION:
            add_result(
                workspace_name,
                "Workspace",
                workspace_name,
                workspace_id,
                current_runtime,
                TARGET_RUNTIME,
                "Planned",
                "Dry run: no API mutation was sent.",
            )
            return

        response = call_api(
            "patch",
            f"/v1/workspaces/{workspace_id}/spark/settings",
            json={"environment": {"runtimeVersion": TARGET_RUNTIME}},
        )

        if response.status_code != 200:
            add_result(
                workspace_name,
                "Workspace",
                workspace_name,
                workspace_id,
                current_runtime,
                current_runtime,
                "Failed",
                response_error(response),
            )
            return

        verified_runtime = get_workspace_runtime(workspace_id)
        status = "Verified" if verified_runtime == TARGET_RUNTIME else "Failed"
        detail = "" if status == "Verified" else "Post-update verification failed."
        add_result(
            workspace_name,
            "Workspace",
            workspace_name,
            workspace_id,
            current_runtime,
            verified_runtime,
            status,
            detail,
        )
    except Exception as exception:
        add_result(
            workspace_name,
            "Workspace",
            workspace_name,
            workspace_id,
            None,
            None,
            "Failed",
            str(exception),
        )


def migrate_environment(workspace_name, workspace_id, environment):
    environment_id = environment["id"]
    environment_name = environment["displayName"]

    try:
        metadata = get_environment_metadata(workspace_id, environment_id)
        current_publish_state = publish_state(metadata)

        if current_publish_state in {"Running", "Waiting", "Cancelling"}:
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                None,
                None,
                "Skipped",
                f"Publish state is {current_publish_state}.",
            )
            return

        published_compute = get_environment_compute(
            workspace_id, environment_id, staging=False
        )
        current_runtime = published_compute.get("runtimeVersion")

        if current_runtime != SOURCE_RUNTIME:
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                current_runtime,
                current_runtime,
                "Skipped",
                f"Only Runtime {SOURCE_RUNTIME} is migrated.",
            )
            return

        staging_compute = get_environment_compute(
            workspace_id, environment_id, staging=True
        )

        if comparable_compute(staging_compute) != comparable_compute(published_compute):
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                current_runtime,
                current_runtime,
                "Skipped",
                "Unpublished Environment changes require manual review.",
            )
            return

        if not EXECUTE_MIGRATION:
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                current_runtime,
                TARGET_RUNTIME,
                "Planned",
                "Dry run: no API mutation was sent.",
            )
            return

        update_response = call_api(
            "patch",
            f"/v1/workspaces/{workspace_id}/environments/"
            f"{environment_id}/staging/sparkcompute?beta=false",
            json={"runtimeVersion": TARGET_RUNTIME},
        )

        if update_response.status_code != 200:
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                current_runtime,
                current_runtime,
                "Failed",
                response_error(update_response),
            )
            return

        publish_response = call_api(
            "post",
            f"/v1/workspaces/{workspace_id}/environments/"
            f"{environment_id}/staging/publish?beta=false",
        )

        if publish_response.status_code not in {200, 202}:
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                current_runtime,
                current_runtime,
                "Failed",
                response_error(publish_response),
            )
            return

        final_publish_state = wait_for_environment_publish(
            workspace_id, environment_id
        )

        if final_publish_state != "Success":
            add_result(
                workspace_name,
                "Environment",
                environment_name,
                environment_id,
                current_runtime,
                current_runtime,
                "Failed",
                f"Publish ended with state {final_publish_state}.",
            )
            return

        verified_compute = get_environment_compute(
            workspace_id, environment_id, staging=False
        )
        verified_runtime = verified_compute.get("runtimeVersion")
        status = "Verified" if verified_runtime == TARGET_RUNTIME else "Failed"
        detail = "" if status == "Verified" else "Post-publish verification failed."
        add_result(
            workspace_name,
            "Environment",
            environment_name,
            environment_id,
            current_runtime,
            verified_runtime,
            status,
            detail,
        )
    except Exception as exception:
        add_result(
            workspace_name,
            "Environment",
            environment_name,
            environment_id,
            None,
            None,
            "Failed",
            str(exception),
        )


selected_workspaces = select_workspaces()

print(f"Mode: {'EXECUTE' if EXECUTE_MIGRATION else 'DRY RUN'}")
print(f"Target runtime: {TARGET_RUNTIME}")
print(f"Selected workspaces: {len(selected_workspaces)}")

for _, workspace in selected_workspaces.iterrows():
    workspace_id = workspace["Id"]
    workspace_name = workspace["Name"]

    print(f"Processing workspace: {workspace_name}")
    migrate_workspace(workspace_name, workspace_id)

    try:
        environments = list_environments(workspace_id)
    except Exception as exception:
        add_result(
            workspace_name,
            "Environment",
            None,
            None,
            None,
            None,
            "Failed",
            f"Unable to list environments: {exception}",
        )
        continue

    for environment in environments:
        migrate_environment(
            workspace_name, workspace_id, environment
        )

migration_df = pd.DataFrame(migration_results)
display(migration_df)

if not migration_df.empty:
    display(
        migration_df.groupby(["Scope", "Status"], dropna=False)
        .size()
        .reset_index(name="Count")
    )